# HuggingFace 모델 4종 일괄 이어받기

snap2read에 필요한 PaddleOCR 모델 **4개 저장소를 자동으로 순회**하며 전부 받는 노트북.
- **이어받기**: 중단했다가 다시 실행하면 이어서 받음 (HTTP Range)
- **진행률**: 파일별 진행바 + 4개 저장소 합산 전체 진행률 + 속도/남은시간
- 직접 연결(프록시 없이) 다운로드합니다.

대상: `PP-OCRv6_medium_det`, `PP-OCRv6_medium_rec`, `PP-LCNet_x1_0_textline_ori`, `PP-LCNet_x1_0_doc_ori`
→ 각각 `./<모델명>/` 폴더로 저장 (snap2read 서버가 바로 인식하는 위치)


In [ ]:
import os
import time
import json
import urllib.request

# ============ 대상 저장소 4개 (자동 순회) ============
REPOS = [
    "PaddlePaddle/PP-OCRv6_medium_det",
    "PaddlePaddle/PP-OCRv6_medium_rec",
    "PaddlePaddle/PP-LCNet_x1_0_textline_ori",
    "PaddlePaddle/PP-LCNet_x1_0_doc_ori",
]
# ==================================================

CHUNK = 1024 * 1024  # 1MB


In [ ]:
def format_time(seconds):
    if seconds < 60:
        return f"{seconds:.0f}초"
    elif seconds < 3600:
        return f"{seconds/60:.1f}분"
    else:
        return f"{seconds/3600:.1f}시간"


def save_dir(repo):
    """저장 폴더는 저장소 이름과 동일하게 (./<모델명>/)"""
    return os.path.join(".", repo.split("/")[-1])


def fetch_file_list(repo):
    """HuggingFace API로 파일 목록 가져오기"""
    api = f"https://huggingface.co/api/models/{repo}/tree/main"
    req = urllib.request.Request(api)
    req.add_header("User-Agent", "Mozilla/5.0")
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return sorted(
        [f for f in data if f.get("type") == "file"],
        key=lambda x: x.get("size", 0),
    )


def download_file(repo, file_info, global_state):
    """단일 파일 이어받기 (중단 시 재실행하면 이어서 받음)"""
    filename = file_info["path"]
    file_size = file_info["size"]
    d = save_dir(repo)
    os.makedirs(d, exist_ok=True)
    save_path = os.path.join(d, filename)
    url = f"https://huggingface.co/{repo}/resolve/main/{filename}"

    # 이미 완료된 경우 스킵
    if os.path.exists(save_path) and os.path.getsize(save_path) == file_size:
        print(f"  ✅ 이미 완료: {filename}")
        return True

    # 부분 다운로드 크기
    downloaded = os.path.getsize(save_path) if os.path.exists(save_path) else 0

    # 이어받기 요청
    req = urllib.request.Request(url)
    req.add_header("User-Agent", "Mozilla/5.0")
    if downloaded > 0:
        req.add_header("Range", f"bytes={downloaded}-")

    try:
        resp = urllib.request.urlopen(req, timeout=60)
    except Exception as e:
        print(f"  ❌ 연결 실패: {e}")
        return False

    # Range 응답 확인
    if downloaded > 0 and resp.status != 206:
        print("  ⚠️ 이어받기 불가, 처음부터 다시 받습니다.")
        downloaded = 0

    mode = "ab" if downloaded > 0 and resp.status == 206 else "wb"
    start_time = time.time()
    last_print_time = start_time
    last_downloaded = downloaded

    if downloaded > 0:
        print(f"  📥 이어받기: {downloaded/1024**3:.2f} GB 부터")
    else:
        print(f"  📥 새로 받기: {filename}")

    try:
        with open(save_path, mode) as f:
            while True:
                chunk = resp.read(CHUNK)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                global_state["done"] += len(chunk)

                # 진행률 표시 (0.5초마다)
                now = time.time()
                if now - last_print_time >= 0.5:
                    speed = ((downloaded - last_downloaded) / (now - last_print_time)
                             if now > last_print_time else 0)
                    file_progress = downloaded / file_size * 100
                    global_progress = global_state["done"] / global_state["total"] * 100
                    remaining = (file_size - downloaded) / speed if speed > 0 else 0

                    bar_len = 30
                    filled = int(bar_len * file_progress / 100)
                    bar = "█" * filled + "░" * (bar_len - filled)

                    print(
                        f"\r  [{bar}] {file_progress:.1f}% | "
                        f"{downloaded/1024**3:.2f}/{file_size/1024**3:.2f} GB | "
                        f"{speed/1024**2:.1f} MB/s | "
                        f"전체: {global_progress:.1f}% | "
                        f"남은: {format_time(remaining)}",
                        end="", flush=True,
                    )
                    last_print_time = now
                    last_downloaded = downloaded

    except KeyboardInterrupt:
        resp.close()
        print(f"\n  ⏸️ 중단됨. {downloaded/1024**3:.2f} GB 까지 받음.")
        return False
    except Exception as e:
        resp.close()
        print(f"\n  ❌ 오류: {e}")
        return False

    resp.close()
    print(f"\n  ✅ 완료: {filename} ({downloaded/1024**3:.2f} GB)")
    return True

In [ ]:
# 4개 저장소 파일 목록 수집 + 전체 계획 수립
plan = []
for repo in REPOS:
    print(f"📂 {repo} 파일 목록 조회 중...")
    try:
        files = fetch_file_list(repo)
    except Exception as e:
        print(f"  ❌ 목록 조회 실패: {e} — 이 저장소는 건너뜁니다.")
        continue
    plan.append((repo, files))
    print(f"  → {len(files)}개 파일, {sum(f['size'] for f in files)/1024**3:.2f} GB")

global_state = {
    "total": sum(f["size"] for _, files in plan for f in files),
}

# 이미 완료된 크기 반영 (이어받기 기준점)
done = 0
for repo, files in plan:
    for f in files:
        p = os.path.join(save_dir(repo), f["path"])
        if os.path.exists(p) and os.path.getsize(p) == f["size"]:
            done += f["size"]
global_state["done"] = done

print(f"\n총 {len(plan)}개 저장소, {global_state['total']/1024**3:.2f} GB | "
      f"이미 완료 {done/1024**3:.2f} GB")

# 전체 다운로드 실행 (저장소 순회)
overall_start = time.time()
for r, (repo, files) in enumerate(plan, 1):
    print("\n" + "=" * 70)
    print(f"🚀 [{r}/{len(plan)}] {repo}")
    print("=" * 70)
    for i, f in enumerate(files, 1):
        print(f"\n[{i}/{len(files)}] {f['path']} ({f['size']/1024**3:.2f} GB)")
        download_file(repo, f, global_state)

# 최종 결과
elapsed = time.time() - overall_start
print("\n" + "=" * 70)
print("🎉 전체 다운로드 완료!")
print(f"   위치: {os.path.abspath('.')}")
print(f"   소요 시간: {format_time(elapsed)}")
print("=" * 70)